<a href="https://colab.research.google.com/github/sharath7-dev/robust-multilingual-intent-classification/blob/main/notebooks/phase1_dataset_verification_tfidf.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Robust and Calibrated Multilingual Intent Classification

## Phase 1 — Dataset Verification and TF-IDF Baseline

### Goal
Verify the multilingual dataset, construct leakage-safe known and unseen intent groups, and establish a reproducible classical baseline without evaluating the test set.

### Configuration
- Dataset: MASSIVE 1.1
- Languages: English (`en-US`), Hindi (`hi-IN`), Kannada (`kn-IN`)
- Released examples: 49,563
- Original intents: 60
- Known intents used for training: 50
- Validation-OOS intents: 5
- Test-OOS intents: 5
- Random seed: 42

### Verified data integrity
- Official train, development, and test partitions preserved
- No missing utterances or intent labels
- No duplicate IDs within a locale
- Parallel examples remain in the same partition and intent across languages
- No held-out OOS intent appears in training

### TF-IDF validation baseline
The baseline combines word and character TF-IDF features with logistic regression.

| Evaluation group | Accuracy | Macro-F1 |
|---|---:|---:|
| Overall validation | 0.8369 | 0.8210 |
| English | 0.8594 | 0.8394 |
| Hindi | 0.8308 | 0.8140 |
| Kannada | 0.8205 | 0.8043 |

Training completed in 61.8 seconds. The solver converged after 115 iterations. The final test set has not been evaluated.

In [1]:
import sys
import platform

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())

try:
    import torch
    print("PyTorch:", torch.__version__)
    print("GPU available:", torch.cuda.is_available())
    print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")
except Exception as error:
    print("PyTorch check failed:", error)

Python: 3.13.16
Platform: Linux-6.6.122+-x86_64-with-glibc2.39
PyTorch: 2.11.0+cu130
GPU available: True
GPU: Tesla T4


In [3]:
%cd /content
!git clone https://github.com/sharath7-dev/robust-multilingual-intent-classification.git
%cd /content/robust-multilingual-intent-classification
!git status

/content
Cloning into 'robust-multilingual-intent-classification'...
remote: Enumerating objects: 4, done.
remote: Counting objects: 100% (4/4), done.
remote: Compressing objects: 100% (4/4), done.
remote: Total 4 (delta 0), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (4/4), done.
/content/robust-multilingual-intent-classification
On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean


In [4]:
from pathlib import Path

repo = Path("/content/robust-multilingual-intent-classification")

folders = [
    "notebooks",
    "src",
    "configs",
    "results/metrics",
    "results/figures",
    "docs",
    "ui",
]

for folder in folders:
    path = repo / folder
    path.mkdir(parents=True, exist_ok=True)
    (path / ".gitkeep").touch()

extra_gitignore = """

# Project data and trained artifacts
data/
models/
checkpoints/
*.pt
*.pth
*.ckpt
*.safetensors
*.bin

# Secrets and experiment services
.env
.env.*
wandb/
mlruns/

# Generated temporary outputs
*.log
.ipynb_checkpoints/
"""

gitignore = repo / ".gitignore"
existing = gitignore.read_text(encoding="utf-8") if gitignore.exists() else ""

if "# Project data and trained artifacts" not in existing:
    gitignore.write_text(existing.rstrip() + extra_gitignore, encoding="utf-8")

print("Created project structure:")
for folder in folders:
    print(" -", folder)

print("\nGit status:")

Created project structure:
 - notebooks
 - src
 - configs
 - results/metrics
 - results/figures
 - docs
 - ui

Git status:


In [5]:
!git status --short

 M .gitignore
?? configs/
?? docs/
?? notebooks/
?? results/
?? src/
?? ui/


In [6]:
from google.colab import userdata
from pathlib import Path
import os
import subprocess
import tempfile

repo = Path("/content/robust-multilingual-intent-classification")
token = userdata.get("GITHUB_TOKEN")

if not token:
    raise RuntimeError("GITHUB_TOKEN is unavailable. Check Notebook access in Colab Secrets.")

def run_git(*args, env=None):
    return subprocess.run(
        ["git", *args],
        cwd=repo,
        env=env,
        text=True,
        check=True,
    )

run_git("config", "user.name", "sharath7-dev")
run_git("config", "user.email", "sharath7-dev@users.noreply.github.com")

run_git("add", ".gitignore", "configs", "docs", "notebooks", "results", "src", "ui")
run_git("commit", "-m", "chore: initialize project structure and gitignore")

askpass_path = None

try:
    with tempfile.NamedTemporaryFile(
        mode="w", suffix=".sh", delete=False, encoding="utf-8"
    ) as askpass:
        askpass.write(
            '#!/bin/sh\n'
            'case "$1" in\n'
            '  *Username*) echo "x-access-token" ;;\n'
            '  *) echo "$GITHUB_TOKEN" ;;\n'
            'esac\n'
        )
        askpass_path = askpass.name

    os.chmod(askpass_path, 0o700)

    push_env = os.environ.copy()
    push_env["GITHUB_TOKEN"] = token
    push_env["GIT_ASKPASS"] = askpass_path
    push_env["GIT_TERMINAL_PROMPT"] = "0"

    run_git("push", "origin", "main", env=push_env)

finally:
    if askpass_path and os.path.exists(askpass_path):
        os.remove(askpass_path)

print("\nLatest commit:")
run_git("log", "-1", "--oneline")

print("\nWorking tree:")
run_git("status", "--short")


Latest commit:

Working tree:


CompletedProcess(args=['git', 'status', '--short'], returncode=0)

In [7]:
import subprocess
from pathlib import Path

repo = Path("/content/robust-multilingual-intent-classification")

def show_git(*args):
    result = subprocess.run(
        ["git", *args],
        cwd=repo,
        text=True,
        capture_output=True,
        check=True,
    )
    print(result.stdout.strip() or "(no output)")

print("Latest local commit:")
show_git("log", "-1", "--oneline")

print("\nRemote tracking status:")
show_git("status", "--short", "--branch")

Latest local commit:
087b5c7 chore: initialize project structure and gitignore

Remote tracking status:
## main...origin/main


In [8]:
from pathlib import Path
from urllib.request import urlretrieve

repo = Path("/content/robust-multilingual-intent-classification")
download_dir = repo / "data"
download_dir.mkdir(exist_ok=True)

url = "https://amazon-massive-nlu-dataset.s3.amazonaws.com/amazon-massive-dataset-1.1.tar.gz"
archive = download_dir / "amazon-massive-dataset-1.1.tar.gz"

if not archive.exists():
    print("Downloading MASSIVE 1.1...")
    urlretrieve(url, archive)
else:
    print("Archive already exists.")

size_mb = archive.stat().st_size / (1024 * 1024)

print("File:", archive.name)
print(f"Size: {size_mb:.2f} MB")
print("Download verified:", archive.exists() and archive.stat().st_size > 0)

File: amazon-massive-dataset-1.1.tar.gz
Size: 38.39 MB
Download verified: True


In [9]:
import tarfile
from pathlib import Path

repo = Path("/content/robust-multilingual-intent-classification")
data_dir = repo / "data"
archive = data_dir / "amazon-massive-dataset-1.1.tar.gz"

with tarfile.open(archive, "r:gz") as tar:
    tar.extractall(data_dir, filter="data")

required_files = {}

for locale in ["en-US", "hi-IN", "kn-IN"]:
    matches = list(data_dir.rglob(f"{locale}.jsonl"))
    required_files[locale] = matches

print("Required language files:")

for locale, matches in required_files.items():
    if len(matches) == 1:
        print(f"{locale}: FOUND — {matches[0]}")
    else:
        print(f"{locale}: ERROR — found {len(matches)} matching files")

Required language files:
en-US: FOUND — /content/robust-multilingual-intent-classification/data/1.1/data/en-US.jsonl
hi-IN: FOUND — /content/robust-multilingual-intent-classification/data/1.1/data/hi-IN.jsonl
kn-IN: FOUND — /content/robust-multilingual-intent-classification/data/1.1/data/kn-IN.jsonl


In [10]:
import json
import pandas as pd
from pathlib import Path

data_root = Path(
    "/content/robust-multilingual-intent-classification/data/1.1/data"
)

locales = ["en-US", "hi-IN", "kn-IN"]
frames = []

for locale in locales:
    file_path = data_root / f"{locale}.jsonl"

    with file_path.open("r", encoding="utf-8") as file:
        records = [json.loads(line) for line in file]

    df_locale = pd.DataFrame(records)
    frames.append(df_locale)

    print(f"\n--- {locale} ---")
    print("Rows:", len(df_locale))
    print("Partitions:", df_locale["partition"].value_counts().to_dict())
    print("Unique intents:", df_locale["intent"].nunique())
    print("Missing utterances:", df_locale["utt"].isna().sum())
    print("Missing intents:", df_locale["intent"].isna().sum())
    print("Duplicate IDs:", df_locale["id"].duplicated().sum())
    print("Sample:", df_locale["utt"].iloc[0])

df = pd.concat(frames, ignore_index=True)

print("\nCombined rows:", len(df))
print("Columns:", df.columns.tolist())
print("Locales:", sorted(df["locale"].unique().tolist()))


--- en-US ---
Rows: 16521
Partitions: {'train': 11514, 'test': 2974, 'dev': 2033}
Unique intents: 60
Missing utterances: 0
Missing intents: 0
Duplicate IDs: 0
Sample: wake me up at five am this week

--- hi-IN ---
Rows: 16521
Partitions: {'train': 11514, 'test': 2974, 'dev': 2033}
Unique intents: 60
Missing utterances: 0
Missing intents: 0
Duplicate IDs: 0
Sample: मुझे इस सप्ताह सुबह पांच बजे जगा दो

--- kn-IN ---
Rows: 16521
Partitions: {'train': 11514, 'test': 2974, 'dev': 2033}
Unique intents: 60
Missing utterances: 0
Missing intents: 0
Duplicate IDs: 0
Sample: ಈ ವಾರ ಬೆಳಗ್ಗೆ ಐದು ಗಂಟೆಗೆ ನನ್ನನ್ನು ಎದ್ದೇಳಿ

Combined rows: 49563
Columns: ['id', 'locale', 'partition', 'scenario', 'intent', 'utt', 'annot_utt', 'worker_id', 'slot_method', 'judgments']
Locales: ['en-US', 'hi-IN', 'kn-IN']


In [11]:
# Each shared utterance ID should remain in the same partition
# and have the same intent across all three languages.

id_checks = df.groupby("id").agg(
    locale_count=("locale", "nunique"),
    partition_count=("partition", "nunique"),
    intent_count=("intent", "nunique"),
)

incomplete_ids = (id_checks["locale_count"] != 3).sum()
partition_conflicts = (id_checks["partition_count"] != 1).sum()
intent_conflicts = (id_checks["intent_count"] != 1).sum()

print("Unique shared IDs:", len(id_checks))
print("IDs missing one or more selected languages:", incomplete_ids)
print("IDs assigned to multiple partitions:", partition_conflicts)
print("IDs assigned to multiple intents:", intent_conflicts)

passed = (
    incomplete_ids == 0
    and partition_conflicts == 0
    and intent_conflicts == 0
)

print("Parallel split verification passed:", passed)

Unique shared IDs: 16521
IDs missing one or more selected languages: 0
IDs assigned to multiple partitions: 0
IDs assigned to multiple intents: 0
Parallel split verification passed: True


In [12]:
import json
import random
from pathlib import Path

repo = Path("/content/robust-multilingual-intent-classification")

all_intents = sorted(df["intent"].unique().tolist())

rng = random.Random(42)
held_out_intents = sorted(rng.sample(all_intents, 10))

validation_oos_intents = held_out_intents[:5]
test_oos_intents = held_out_intents[5:]
known_intents = sorted(set(all_intents) - set(held_out_intents))

intent_split = {
    "seed": 42,
    "known_intents": known_intents,
    "validation_oos_intents": validation_oos_intents,
    "test_oos_intents": test_oos_intents,
}

config_path = repo / "configs" / "intent_split.json"
config_path.write_text(
    json.dumps(intent_split, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

print("Known intents:", len(known_intents))
print("Validation OOS intents:", validation_oos_intents)
print("Test OOS intents:", test_oos_intents)
print("Overlap between OOS groups:",
      set(validation_oos_intents) & set(test_oos_intents))
print("Saved:", config_path)

Known intents: 50
Validation OOS intents: ['alarm_remove', 'audio_volume_up', 'calendar_query', 'calendar_remove', 'email_addcontact']
Test OOS intents: ['email_query', 'email_sendemail', 'play_music', 'qa_stock', 'transport_taxi']
Overlap between OOS groups: set()
Saved: /content/robust-multilingual-intent-classification/configs/intent_split.json


In [13]:
known_set = set(known_intents)
validation_oos_set = set(validation_oos_intents)
test_oos_set = set(test_oos_intents)

train_known = df[
    (df["partition"] == "train") &
    (df["intent"].isin(known_set))
].copy()

validation_known = df[
    (df["partition"] == "dev") &
    (df["intent"].isin(known_set))
].copy()

validation_oos = df[
    (df["partition"] == "dev") &
    (df["intent"].isin(validation_oos_set))
].copy()

test_known = df[
    (df["partition"] == "test") &
    (df["intent"].isin(known_set))
].copy()

test_oos = df[
    (df["partition"] == "test") &
    (df["intent"].isin(test_oos_set))
].copy()

splits = {
    "train_known": train_known,
    "validation_known": validation_known,
    "validation_oos": validation_oos,
    "test_known": test_known,
    "test_oos": test_oos,
}

print("Split sizes:")
for name, split_df in splits.items():
    print(f"\n{name}: {len(split_df)}")
    print(split_df["locale"].value_counts().sort_index().to_dict())

training_leakage = set(train_known["intent"]) & (
    validation_oos_set | test_oos_set
)

print("\nHeld-out intents found in training:", training_leakage)
print("Training has exactly 50 intents:", train_known["intent"].nunique() == 50)
print("Verification passed:",
      not training_leakage and train_known["intent"].nunique() == 50)

Split sizes:

train_known: 26193
{'en-US': 8731, 'hi-IN': 8731, 'kn-IN': 8731}

validation_known: 4629
{'en-US': 1543, 'hi-IN': 1543, 'kn-IN': 1543}

validation_oos: 540
{'en-US': 180, 'hi-IN': 180, 'kn-IN': 180}

test_known: 6831
{'en-US': 2277, 'hi-IN': 2277, 'kn-IN': 2277}

test_oos: 1374
{'en-US': 458, 'hi-IN': 458, 'kn-IN': 458}

Held-out intents found in training: set()
Training has exactly 50 intents: True
Verification passed: True


In [14]:
import json
from pathlib import Path

repo = Path("/content/robust-multilingual-intent-classification")

verification_report = {
    "dataset": "MASSIVE 1.1",
    "source_archive": "amazon-massive-dataset-1.1.tar.gz",
    "archive_size_mb": 38.39,
    "locales": ["en-US", "hi-IN", "kn-IN"],
    "rows_per_locale": 16521,
    "combined_rows": 49563,
    "official_split_counts_per_locale": {
        "train": 11514,
        "dev": 2033,
        "test": 2974,
    },
    "unique_intents": 60,
    "missing_utterances": 0,
    "missing_intents": 0,
    "duplicate_ids_per_locale": 0,
    "parallel_split_conflicts": 0,
    "parallel_intent_conflicts": 0,
    "known_intents": 50,
    "validation_oos_intents": validation_oos_intents,
    "test_oos_intents": test_oos_intents,
    "experimental_split_sizes": {
        name: {
            "total": len(split_df),
            "by_locale": split_df["locale"].value_counts().sort_index().to_dict(),
        }
        for name, split_df in splits.items()
    },
    "training_oos_leakage": False,
    "verification_passed": True,
}

report_path = repo / "results" / "metrics" / "dataset_verification.json"
report_path.write_text(
    json.dumps(verification_report, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

print(report_path.read_text(encoding="utf-8"))
print("\nGit status:")

{
  "dataset": "MASSIVE 1.1",
  "source_archive": "amazon-massive-dataset-1.1.tar.gz",
  "archive_size_mb": 38.39,
  "locales": [
    "en-US",
    "hi-IN",
    "kn-IN"
  ],
  "rows_per_locale": 16521,
  "combined_rows": 49563,
  "official_split_counts_per_locale": {
    "train": 11514,
    "dev": 2033,
    "test": 2974
  },
  "unique_intents": 60,
  "missing_utterances": 0,
  "missing_intents": 0,
  "duplicate_ids_per_locale": 0,
  "parallel_split_conflicts": 0,
  "parallel_intent_conflicts": 0,
  "known_intents": 50,
  "validation_oos_intents": [
    "alarm_remove",
    "audio_volume_up",
    "calendar_query",
    "calendar_remove",
    "email_addcontact"
  ],
  "test_oos_intents": [
    "email_query",
    "email_sendemail",
    "play_music",
    "qa_stock",
    "transport_taxi"
  ],
  "experimental_split_sizes": {
    "train_known": {
      "total": 26193,
      "by_locale": {
        "en-US": 8731,
        "hi-IN": 8731,
        "kn-IN": 8731
      }
    },
    "validation_known": {

In [15]:
!git status --short

?? configs/intent_split.json
?? results/metrics/dataset_verification.json


In [16]:
from google.colab import userdata
from pathlib import Path
import os
import subprocess
import tempfile

repo = Path("/content/robust-multilingual-intent-classification")
token = userdata.get("GITHUB_TOKEN")

subprocess.run(
    ["git", "add",
     "configs/intent_split.json",
     "results/metrics/dataset_verification.json"],
    cwd=repo,
    check=True,
)

subprocess.run(
    ["git", "commit", "-m",
     "data: verify MASSIVE multilingual splits and freeze OOS intents"],
    cwd=repo,
    check=True,
)

askpass_path = None

try:
    with tempfile.NamedTemporaryFile(
        mode="w", suffix=".sh", delete=False, encoding="utf-8"
    ) as askpass:
        askpass.write(
            '#!/bin/sh\n'
            'case "$1" in\n'
            '  *Username*) echo "x-access-token" ;;\n'
            '  *) echo "$GITHUB_TOKEN" ;;\n'
            'esac\n'
        )
        askpass_path = askpass.name

    os.chmod(askpass_path, 0o700)

    env = os.environ.copy()
    env["GITHUB_TOKEN"] = token
    env["GIT_ASKPASS"] = askpass_path
    env["GIT_TERMINAL_PROMPT"] = "0"

    subprocess.run(
        ["git", "push", "origin", "main"],
        cwd=repo,
        env=env,
        check=True,
    )

finally:
    if askpass_path and os.path.exists(askpass_path):
        os.remove(askpass_path)

latest = subprocess.run(
    ["git", "log", "-1", "--oneline"],
    cwd=repo,
    text=True,
    capture_output=True,
    check=True,
)

status = subprocess.run(
    ["git", "status", "--short", "--branch"],
    cwd=repo,
    text=True,
    capture_output=True,
    check=True,
)

print("Latest commit:", latest.stdout.strip())
print("Status:", status.stdout.strip())

Latest commit: a7accaa data: verify MASSIVE multilingual splits and freeze OOS intents
Status: ## main...origin/main


In [17]:
import subprocess
from pathlib import Path

repo = Path("/content/robust-multilingual-intent-classification")

pull = subprocess.run(
    ["git", "pull", "--ff-only", "origin", "main"],
    cwd=repo,
    text=True,
    capture_output=True,
    check=True,
)

status = subprocess.run(
    ["git", "status", "--short", "--branch"],
    cwd=repo,
    text=True,
    capture_output=True,
    check=True,
)

print(pull.stdout.strip())
print(status.stdout.strip())

Updating a7accaa..368eab3
Fast-forward
 notebooks/phase1_dataset_verification_tfidf.ipynb | 1025 +++++++++++++++++++++
 1 file changed, 1025 insertions(+)
 create mode 100644 notebooks/phase1_dataset_verification_tfidf.ipynb
## main...origin/main


In [18]:
import time
import numpy as np
from sklearn.pipeline import FeatureUnion, Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

tfidf_features = FeatureUnion([
    (
        "word",
        TfidfVectorizer(
            analyzer="word",
            ngram_range=(1, 2),
            min_df=2,
            max_features=30_000,
            sublinear_tf=True,
        ),
    ),
    (
        "character",
        TfidfVectorizer(
            analyzer="char_wb",
            ngram_range=(3, 5),
            min_df=2,
            max_features=50_000,
            sublinear_tf=True,
        ),
    ),
])

tfidf_model = Pipeline([
    ("features", tfidf_features),
    (
        "classifier",
        LogisticRegression(
            solver="saga",
            max_iter=300,
            C=4.0,
            random_state=42,
            n_jobs=-1,
        ),
    ),
])

start_time = time.time()

tfidf_model.fit(
    train_known["utt"],
    train_known["intent"],
)

training_seconds = time.time() - start_time

validation_predictions = tfidf_model.predict(validation_known["utt"])

validation_accuracy = accuracy_score(
    validation_known["intent"],
    validation_predictions,
)

validation_macro_f1 = f1_score(
    validation_known["intent"],
    validation_predictions,
    average="macro",
)

print(f"Training time: {training_seconds:.1f} seconds")
print(f"Validation accuracy: {validation_accuracy:.4f}")
print(f"Validation macro-F1: {validation_macro_f1:.4f}")

print("\nPer-language validation results:")

for locale in ["en-US", "hi-IN", "kn-IN"]:
    mask = validation_known["locale"] == locale

    locale_accuracy = accuracy_score(
        validation_known.loc[mask, "intent"],
        validation_predictions[mask],
    )

    locale_macro_f1 = f1_score(
        validation_known.loc[mask, "intent"],
        validation_predictions[mask],
        average="macro",
    )

    print(
        f"{locale}: accuracy={locale_accuracy:.4f}, "
        f"macro-F1={locale_macro_f1:.4f}"
    )

Training time: 61.8 seconds
Validation accuracy: 0.8369
Validation macro-F1: 0.8210

Per-language validation results:
en-US: accuracy=0.8594, macro-F1=0.8394
hi-IN: accuracy=0.8308, macro-F1=0.8140
kn-IN: accuracy=0.8205, macro-F1=0.8043


In [19]:
import json
import joblib
import sklearn
from pathlib import Path

repo = Path("/content/robust-multilingual-intent-classification")

per_language_results = {}

for locale in ["en-US", "hi-IN", "kn-IN"]:
    mask = validation_known["locale"] == locale

    per_language_results[locale] = {
        "accuracy": float(
            accuracy_score(
                validation_known.loc[mask, "intent"],
                validation_predictions[mask],
            )
        ),
        "macro_f1": float(
            f1_score(
                validation_known.loc[mask, "intent"],
                validation_predictions[mask],
                average="macro",
            )
        ),
    }

baseline_metrics = {
    "model": "word-and-character TF-IDF with logistic regression",
    "random_seed": 42,
    "scikit_learn_version": sklearn.__version__,
    "training_examples": len(train_known),
    "validation_examples": len(validation_known),
    "training_seconds": training_seconds,
    "validation_accuracy": float(validation_accuracy),
    "validation_macro_f1": float(validation_macro_f1),
    "per_language": per_language_results,
    "test_set_evaluated": False,
}

metrics_path = repo / "results" / "metrics" / "tfidf_validation_metrics.json"
metrics_path.write_text(
    json.dumps(baseline_metrics, indent=2),
    encoding="utf-8",
)

model_dir = repo / "models"
model_dir.mkdir(exist_ok=True)
model_path = model_dir / "tfidf_baseline.joblib"
joblib.dump(tfidf_model, model_path)

max_iterations_used = int(
    tfidf_model.named_steps["classifier"].n_iter_.max()
)

print("Metrics saved:", metrics_path)
print("Model saved outside Git tracking:", model_path)
print(f"Model size: {model_path.stat().st_size / (1024**2):.2f} MB")
print("Maximum solver iterations used:", max_iterations_used)
print("Reached configured 300-iteration limit:", max_iterations_used >= 300)
print("\nGit status:")

Metrics saved: /content/robust-multilingual-intent-classification/results/metrics/tfidf_validation_metrics.json
Model saved outside Git tracking: /content/robust-multilingual-intent-classification/models/tfidf_baseline.joblib
Model size: 27.56 MB
Maximum solver iterations used: 115
Reached configured 300-iteration limit: False

Git status:


In [20]:
%cd /content/robust-multilingual-intent-classification
!git status --short --untracked-files=all
!git check-ignore -v results/metrics/tfidf_validation_metrics.json || echo "Metrics file is not ignored"

/content/robust-multilingual-intent-classification
?? results/metrics/tfidf_validation_metrics.json
Metrics file is not ignored


In [23]:
%%writefile /content/robust-multilingual-intent-classification/README.md
# Robust and Calibrated Multilingual Intent Classification

A reproducible NLP project studying multilingual intent classification under spelling noise, transliterated input, unseen intents, and uncertainty-aware rejection.

## Current scope

Languages:

- English (`en-US`)
- Hindi (`hi-IN`)
- Kannada (`kn-IN`)

## Dataset

The project uses the public [MASSIVE 1.1 dataset](https://github.com/alexa/massive), licensed under CC BY 4.0. Dataset files are downloaded when required and are not stored in this repository.

Verified statistics:

- 49,563 examples across the selected languages
- 60 original intents
- 50 known training intents
- 5 validation-OOS intents
- 5 separate test-OOS intents
- Official dataset partitions preserved
- No held-out intent leakage into training

The frozen intent groups are stored in [`configs/intent_split.json`](configs/intent_split.json).

## Phase 1 verified baseline

Model: word-and-character TF-IDF with logistic regression.

| Evaluation group | Accuracy | Macro-F1 |
|---|---:|---:|
| Overall validation | 0.8369 | 0.8210 |
| English | 0.8594 | 0.8394 |
| Hindi | 0.8308 | 0.8140 |
| Kannada | 0.8205 | 0.8043 |

Training completed in 61.8 seconds. The solver converged after 115 iterations. The test set has not been evaluated.

## Repository structure

- `configs/`: experiment configurations
- `docs/`: documentation
- `notebooks/`: Colab notebooks
- `results/metrics/`: evaluation metrics
- `results/figures/`: plots
- `src/`: reusable project code
- `ui/`: final interface

Datasets, secrets, environments, and model checkpoints are excluded from Git.

## Current status

- [x] Dataset verification
- [x] Leakage-safe OOS split
- [x] TF-IDF validation baseline
- [ ] XLM-R multilingual model
- [ ] Noise robustness evaluation
- [ ] Transliteration diagnostic
- [ ] OOS detection and calibration
- [ ] Final test evaluation
- [ ] Professional interface

Only verified experimental results are reported.

Overwriting /content/robust-multilingual-intent-classification/README.md


In [24]:
%cd /content/robust-multilingual-intent-classification
!git status --short
!git diff --check
!git diff --stat

/content/robust-multilingual-intent-classification
 M README.md
?? results/metrics/tfidf_validation_metrics.json
 README.md | 68 +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++--
 1 file changed, 66 insertions(+), 2 deletions(-)


In [25]:
from google.colab import userdata
from pathlib import Path
import os
import subprocess
import tempfile

repo = Path("/content/robust-multilingual-intent-classification")
token = userdata.get("GITHUB_TOKEN")

subprocess.run(
    ["git", "add", "README.md",
     "results/metrics/tfidf_validation_metrics.json"],
    cwd=repo,
    check=True,
)

subprocess.run(
    ["git", "commit", "-m",
     "baseline: add verified TF-IDF validation results"],
    cwd=repo,
    check=True,
)

with tempfile.NamedTemporaryFile(
    mode="w", suffix=".sh", delete=False, encoding="utf-8"
) as askpass:
    askpass.write(
        '#!/bin/sh\n'
        'case "$1" in\n'
        '  *Username*) echo "x-access-token" ;;\n'
        '  *) echo "$GITHUB_TOKEN" ;;\n'
        'esac\n'
    )
    askpass_path = askpass.name

try:
    os.chmod(askpass_path, 0o700)

    env = os.environ.copy()
    env["GITHUB_TOKEN"] = token
    env["GIT_ASKPASS"] = askpass_path
    env["GIT_TERMINAL_PROMPT"] = "0"

    subprocess.run(
        ["git", "push", "origin", "main"],
        cwd=repo,
        env=env,
        check=True,
    )
finally:
    os.remove(askpass_path)

latest = subprocess.run(
    ["git", "log", "-1", "--oneline"],
    cwd=repo,
    text=True,
    capture_output=True,
    check=True,
)

status = subprocess.run(
    ["git", "status", "--short", "--branch"],
    cwd=repo,
    text=True,
    capture_output=True,
    check=True,
)

print("Latest commit:", latest.stdout.strip())
print("Status:", status.stdout.strip())

Latest commit: 78f2dfc baseline: add verified TF-IDF validation results
Status: ## main...origin/main
